In [ ]:
import requests, base64, json, time
import pandas as pd
import re

from bs4 import BeautifulSoup

import vertexai
from vertexai.generative_models import GenerativeModel

import musicbrainzngs

import subprocess
from pathlib import Path
import yaml
import os
import shutil

In [ ]:
!gcloud auth application-default login

In [ ]:
#Spotify API credentials
client_id = 'xxx'
client_secret = 'xxx'

#Genius API Token
GENIUS_TOKEN = "xxx"

In [ ]:
def get_spotify_token(client_id, client_secret):
    auth = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    headers = {"Authorization": f"Basic {auth}"}
    data = {"grant_type": "client_credentials"}
    r = requests.post("https://accounts.spotify.com/api/token", headers=headers, data=data)
    r.raise_for_status()
    return r.json()["access_token"]

In [ ]:
token = get_spotify_token(client_id, client_secret)

## Procura artista

In [ ]:
def search_artist(token, artist_name):
    headers = {"Authorization": f"Bearer {token}"}
    params = {"q": artist_name, "type": "artist", "limit": 1}
    r = requests.get("https://api.spotify.com/v1/search", headers=headers, params=params)
    r.raise_for_status()
    items = r.json().get("artists", {}).get("items", [])
    return items if items else None

In [ ]:
divas = ["Madonna", "Beyoncé", "Lady Gaga", "Adele", "Rihanna", "Mariah Carey", "Britney Spears", "Christina Aguilera", "Taylor Swift", "Ariana Grande",
            "Jennifer Lopez", "Kylie Minogue", "Shakira", "Dua Lipa", "Katy Perry", "Rosalía", "Cher", "Lana Del Rey", "Cyndi Lauper", "Billie Eilish",
            "Demi Lovato", "Selena Gomez", "Miley Cyrus", "Camila Cabello", "Fergie", "Olivia Rodrigo", "Sabrina Carpenter", "Tate McRae", "Chappell Roan", 
            "Charli xcx", "Whitney Houston"
         ]

In [ ]:
df_divas_pop = pd.DataFrame()

for diva in divas:
    artist_data = search_artist(token, diva)
    if artist_data:
        row = {
            'id': artist_data[0]['id'],
            'name': artist_data[0]['name'],
            'type': artist_data[0]['type'],
            'genres': artist_data[0]['genres'],
            'popularity': artist_data[0]['popularity'],
            'followers': artist_data[0]['followers']['total'],
            'image_url': artist_data[0]['images'][0]['url'] if artist_data[0].get('images') else None
        }
        df_divas_pop = pd.concat([df_divas_pop, pd.DataFrame([row])], ignore_index=True)
    time.sleep(1)  # To avoid hitting rate limits

In [ ]:
df_divas_pop

## Procura albums

In [ ]:
def get_artist_albums(token, artist_id):
    headers = {"Authorization": f"Bearer {token}"}
    albums = []
    url = f"https://api.spotify.com/v1/artists/{artist_id}/albums"
    params = {"limit": 50, "include_groups": "album", "market": "US"}
    while True:
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        j = r.json()
        albums.extend(j.get("items", []))
        if j.get("next"):
            url = j["next"]
            params = None
            time.sleep(0.1)
        else:
            break
    # dedupe by album name + release_date
    seen = set()
    unique = []
    for a in albums:
        key = (a["name"].lower(), a.get("release_date"))
        if key not in seen:
            seen.add(key); unique.append(a)
    return unique

In [ ]:
df_albums_divas_pop = pd.DataFrame()

for id, diva in df_divas_pop[['id', 'name']].itertuples(index=False):
    album_data = get_artist_albums(token, id)
    for album in album_data:
        if album:
            row = {
                'id': album['id'],
                'diva_id': id,
                'diva_name': diva,
                'name': album['name'],
                'total_tracks': album['total_tracks'],
                'release_date': album['release_date'],
                'image_url': album['images'][0]['url'] if album.get('images') else None
            }
            df_albums_divas_pop = pd.concat([df_albums_divas_pop, pd.DataFrame([row])], ignore_index=True)
    time.sleep(1)  # To avoid hitting rate limits

In [ ]:
df_albums_divas_pop

In [ ]:
df_albums_exclude = pd.read_excel('base_tfm.xlsx', sheet_name='albums')
df_albums_exclude = df_albums_exclude[df_albums_exclude['exclude'] == 'x'][['id']]

In [ ]:
df_albums_divas_pop = df_albums_divas_pop[~df_albums_divas_pop['id'].isin(df_albums_exclude['id'])]
df_albums_divas_pop

## Procura músicas

In [ ]:
def get_album_tracks(token, album_id):
    headers = {"Authorization": f"Bearer {token}"}
    tracks = []
    url = f"https://api.spotify.com/v1/albums/{album_id}/tracks"
    params = {"limit": 50, "market": "US"}
    while True:
        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        j = r.json()
        tracks.extend(j.get("items", []))
        if j.get("next"):
            url = j["next"]; params = None; time.sleep(0.1)
        else: break
    return tracks

In [ ]:
def get_track_info(token, track_id):
    headers = {"Authorization": f"Bearer {token}"}
    url = f"https://api.spotify.com/v1/tracks/{track_id}"
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    time.sleep(0.1)
    return r.json()

In [ ]:
df_tracks_divas_pop = pd.DataFrame()

for id, album, diva_id, diva_name in df_albums_divas_pop[['id', 'name', 'diva_id', 'diva_name']].itertuples(index=False):
    track_data = get_album_tracks(token, id)
    for track in track_data:
        if track:
            more_track_data = get_track_info(token, track['id'])
            row = {
                'id': track['id'],
                'isrc': more_track_data['external_ids'].get('isrc', None),
                'name': track['name'],
                'track_number': track['track_number'],
                'duration_ms': track['duration_ms'],
                'explicit': track['explicit'],
                'popularity': more_track_data.get('popularity', None),
                'external_url': track['external_urls']['spotify'],
                'diva_id': diva_id,
                'diva_name': diva_name,
                'album_id': id,
                'album_name': album
            }
            df_tracks_divas_pop = pd.concat([df_tracks_divas_pop, pd.DataFrame([row])], ignore_index=True)
    time.sleep(1)  # To avoid hitting rate limits

df_tracks_divas_pop

In [ ]:
df_tracks_base_tfm = pd.read_excel('base_tfm.xlsx', sheet_name='tracks')
df_tracks_exclude = df_tracks_base_tfm[df_tracks_base_tfm['exclude'] == 'x'][['id']]

In [ ]:
df_tracks_divas_pop = df_tracks_divas_pop[~df_tracks_divas_pop['id'].isin(df_tracks_exclude['id'])]
df_tracks_divas_pop

In [ ]:
# remove tracks shorter than 60s and reset index
df_tracks_divas_pop = df_tracks_divas_pop[df_tracks_divas_pop['duration_ms'] >= 60000].reset_index(drop=True)
df_tracks_divas_pop

In [ ]:
df_tracks_divas_pop = df_tracks_divas_pop.merge(
    df_tracks_base_tfm[['id', 'name_modified', 'duration']],
    on='id',
    how='left'
).reindex(columns=[
    'id', 'isrc', 'name', 'name_modified', 'track_number', 'duration_ms', 'duration', 'explicit',
    'popularity', 'external_url', 'diva_id', 'diva_name', 'album_id', 'album_name'
])

## Monta df_final

In [ ]:
df_final = df_tracks_divas_pop.merge(
    df_divas_pop[['id', 'popularity', 'followers', 'image_url']]
        .rename(columns={
            'id': 'diva_id',
            'popularity': 'diva_popularity',
            'followers': 'diva_followers',
            'image_url': 'diva_image_url'
        }),
    left_on='diva_id',
    right_on='diva_id'
)

In [ ]:
df_final = df_final.merge(
    df_albums_divas_pop[['id', 'total_tracks', 'release_date', 'image_url']]
        .rename(columns={
            'id': 'album_id',
            'total_tracks': 'album_total_tracks',
            'total_tracks': 'album_total_tracks',
            'image_url': 'album_image_url'
        }),
    left_on='album_id',
    right_on='album_id'
)

## Busca letras en lrclib

In [ ]:
def buscar_letra_lrclib(artista, musica, max_retries=3):
    """
    Busca uma música no LRCLib e retorna apenas:
    { "plainLyrics": "...", "instrumental": true/false }
    
    Com retry logic e tratamento de erros SSL/rede.
    """
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    
    # Cria session com retry strategy
    session = requests.Session()
    retry_strategy = Retry(
        total=max_retries,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    data = None
    
    try:
        # Primeiro tenta o endpoint exato
        url_get = "https://lrclib.net/api/get"
        params_get = {
            "artist_name": artista,
            "track_name": musica
        }
        
        response = session.get(url_get, params=params_get, timeout=10)
        
        if response.status_code == 200 and response.text.strip():
            try:
                data = response.json()
            except ValueError:
                data = None

        # Se não achar nada, tenta o endpoint /search (busca ampla)
        if not data:
            url_search = "https://lrclib.net/api/search"
            params_search = {"q": f"{artista} {musica}"}
            response = session.get(url_search, params=params_search, timeout=10)
            if response.status_code == 200:
                results = response.json()
                if results:
                    data = results[0]  # pega o primeiro resultado
    
    except (requests.exceptions.RequestException, TimeoutError, ConnectionError) as e:
        print(f"⚠️ Erro ao buscar letra para {artista} - {musica}: {type(e).__name__}")
        # Aguarda um pouco antes de retornar None
        time.sleep(1)
        data = None
    
    finally:
        session.close()

    # Se ainda assim não encontrar
    if not data:
        return {"plainLyrics": None, "instrumental": None}

    # Extrai apenas os campos desejados
    resultado = {
        "plainLyrics": data.get("plainLyrics"),
        "instrumental": data.get("instrumental", False)
    }

    return resultado


# 🔹 Exemplo de uso:
buscar_letra_lrclib("Lady Gaga", "Just Dance")

In [ ]:
df_lyrics = df_final.copy()

for i, row in enumerate(df_lyrics.itertuples(index=False)):
    diva_name = row.diva_name
    track_name = row.name_modified
    lyrics = buscar_letra_lrclib(diva_name, track_name)

    df_lyrics.loc[i, 'instrumental_lrclib'] = lyrics.get("instrumental")
    df_lyrics.loc[i, 'lyrics_lrclib'] = lyrics.get("plainLyrics")

    print(f"{diva_name} : {track_name}")

In [ ]:
df_lyrics

### Busca letras no encontradas en Genius

Funciones para buscar la música y la letra

In [ ]:
def get_lyrics(url):
    # 1. Faz a requisição da página
    page = requests.get(url)
    page.raise_for_status()

    # 2. Cria o objeto BeautifulSoup
    soup = BeautifulSoup(page.text, "html.parser")

    # 3. Busca todos os containers de letra (algumas músicas têm vários)
    containers = soup.find_all("div", class_=lambda x: x and x.startswith("Lyrics__Container"))

    # 4. Junta o texto de cada container
    lyrics = "\n".join([c.get_text(separator="\n").strip() for c in containers])

    # Separar a partir de "Read More"
    if "Read More" in lyrics:
        lyrics_clean = lyrics.split("Read More", 1)[1].strip()  # Pega tudo depois de "Read More" e remove espaços extras
    elif "Lyrics" in lyrics:
        lyrics_clean = lyrics.split("Lyrics", 1)[1].strip()  # Pega tudo depois de "Lyrics" e remove espaços extras
    else:
        lyrics_clean = lyrics

    return lyrics_clean

In [ ]:
def search_song(query):
    base_url = "https://api.genius.com/search"
    headers = {"Authorization": f"Bearer {GENIUS_TOKEN}"}
    params = {"q": query}

    try:
        response = requests.get(base_url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()

        hits = data.get("response", {}).get("hits", [])

        if not hits:
            return None
    
        url = hits[0]["result"]["url"]
        return url
    
    except requests.exceptions.RequestException as e:
        print(f"Erro na requisição: {e}")
        return None
    except KeyError:
        print("Erro ao interpretar a resposta da API.")
        return None

Busca músicas que no se encontraron las letras

In [ ]:
df_lyrics_null = df_lyrics[df_lyrics['instrumental'].isnull()].reset_index(drop=True)
df_lyrics_null

In [ ]:
for i, row in enumerate(df_lyrics_null.itertuples(index=False)):
    diva_name = row.diva_name
    track_name = row.name
    url_genius = search_song(f"{diva_name} {track_name}")
    lyrics = get_lyrics(url_genius) if url_genius else None

    df_lyrics_null.loc[i, 'genius_url'] = url_genius
    df_lyrics_null.loc[i, 'genius_lyrics'] = lyrics

    print(f"{diva_name} : {track_name}")

Merge con el dataframe original

In [ ]:
df_lyrics = df_lyrics.merge(df_lyrics_null[['id', 'genius_lyrics']], on='id', how='left')
df_lyrics['lyrics'] = df_lyrics['lyrics'].fillna(df_lyrics['genius_lyrics'])
df_lyrics = df_lyrics.drop('genius_lyrics', axis=1)

In [ ]:
df_lyrics['instrumental'].fillna(0, inplace=True)

In [ ]:
df_lyrics_final = df_lyrics[df_lyrics['instrumental'] == False]

## Integra com Gemini

In [ ]:
# ==========================================================
# CONFIGURAÇÕES
# ==========================================================
PROJECT_ID = "xxx"
LOCATION = "us-central1"       # local padrão do Vertex AI
MODEL_NAME = "gemini-2.5-pro"


vertexai.init(project=PROJECT_ID, location=LOCATION)
model = GenerativeModel(MODEL_NAME)

# ==========================================================
# PROMPT BASE
# ==========================================================
SYSTEM_PROMPT = (
    """
    Eres un analista de letras musicales. Responde en español.
    Proporciona un JSON con los siguientes campos:

    {
        "temas_principales": [máximo 3 palabras clave, separadas por comas], 

        "emocion_alegria": [intensidad de 0 a 1],
        "emocion_tristeza": [intensidad de 0 a 1],
        "emocion_miedo": [intensidad de 0 a 1],
        "emocion_ira": [intensidad de 0 a 1],
        "emocion_sorpresa": [intensidad de 0 a 1],
        "emocion_asco": [intensidad de 0 a 1],

        "emocion_principal": ["alegría", "tristeza", "miedo", "ira", "sorpresa" o "asco", según la intensidad más alta],

        "resumen": [breve descripción de la letra en máximo 50 palabras],

        "tono_general": ["positivo", "neutro" o "negativo", según predominancia emocional], 
        "energia_emocional": ["baja", "media" o "alta", según la intensidad combinada de las emociones básicas],

        "valencia": [valor entre -1 y 1; negativo = emociones desagradables, positivo = emociones agradables, según el Modelo Circumplejo de Afecto (Russell, 1980)],
        "activacion": [valor entre -1 y 1; negativo = calma, positivo = energía/intensidad, según el Modelo Circumplejo de Afecto (Russell, 1980)]
    }

    Reglas:
    - Usa valores numéricos con punto decimal entre 0 y 1 SOLO para las emociones básicas.
    - Usa valores con punto decimal entre -1 y 1 SOLO para valencia y activación.
    - No añadas explicaciones ni texto fuera del JSON.
    - Sé conciso y directo.
    """
)
# ==========================================================
# PROCESSAR LETRAS
resultados = []

for i, row in df_lyrics_final.iterrows():
    musica_id = row["id"]
    musica_name = row["name"]
    diva_name = row["diva_name"]
    dt_release = row["release_date"]
    letra = row["lyrics"]

    print(f"🎵 Processando música {i+1}/{len(df_lyrics_final)} {musica_name} (ID: {musica_id})...")

    try:
        prompt = f"{SYSTEM_PROMPT}\n\nTitulo de la música: {musica_name}\nArtista:{diva_name}\nFecha de lanzamiento del álbum: {dt_release}\nLetra:\n{letra[:10000]}"  # segurança p/ não exceder limite
        resposta = model.generate_content(prompt)
        texto = resposta.candidates[0].content.parts[0].text

        try:
            json_result = json.loads(texto)
        except json.JSONDecodeError:
            json_result = {"erro_parse": texto}

        resultados.append({"id": musica_id, **json_result})
        time.sleep(0.5)  # pausa leve para evitar rate limit
    except Exception as e:
        print(f"⚠️ Erro na música {musica_id}: {e}")
        resultados.append({"id": musica_id, "erro": str(e)})

# ==========================================================
# SALVAR RESULTADO
# ==========================================================
df_resultado = pd.DataFrame(resultados)
df_resultado.to_csv("resultado_analise_letras_v4.csv", index=False)
print("✅ Análise concluída e salva em resultado_analise_letras.csv")


In [ ]:
df_resultado

In [ ]:
def clean_markdown_block(s: str) -> str:
    if s is None:
        return s
    s = s.strip()
    # remove ```json ou ``` no início e ``` no fim
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s, flags=re.IGNORECASE)
    return s.strip()

def reconstruct_from_char_tokens(tokens):
    """Recebe lista de tokens (após split por ',') e reconstrói palavras."""
    words = []
    current = ""
    for t in tokens:
        s = t.strip()
        if s == "":
            # separador entre palavras (espaço representado por token vazio)
            if current:
                words.append(current)
                current = ""
            # se current vazio, ignora
            continue
        # se token é um único caractere (letra)
        if len(s) == 1:
            current += s
        else:
            # token já parece uma palavra completa
            if current:
                words.append(current)
                current = ""
            words.append(s)
    if current:
        words.append(current)
    # remove vazios e normalize espaços
    words = [w.strip() for w in words if w and w.strip()]
    return words

def normalize_field(value):
    """
    value: pode ser lista, string JSON, string com vírgulas, ou string com letras separadas por vírgulas.
    Retorna: string com palavras separadas por ', ' (ou None se não der pra extrair).
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    # se já for lista
    if isinstance(value, list):
        return ", ".join([str(x).strip() for x in value if x is not None])

    # se for string, limpa markdown e tenta parsear JSON
    if isinstance(value, str):
        s = clean_markdown_block(value)

        # 1) tenta interpretar s como JSON (ex: '["a","b"]' ou '{"k": ...}')
        try:
            parsed = json.loads(s)
            # se é lista, junta
            if isinstance(parsed, list):
                return ", ".join([str(x).strip() for x in parsed if x is not None])
            # se é string (número de camadas), trate abaixo
            if isinstance(parsed, str):
                s = parsed
            # se for outro tipo (ex: número), converte para string
            else:
                return str(parsed)
        except Exception:
            # não é JSON, continua com s como string
            pass

        # 2) split por vírgula e analisa tokens
        tokens = [t for t in [t.strip() for t in s.split(",")]]

        # conta tokens que são single-char (letras)
        single_char_count = sum(1 for t in tokens if len(t) == 1)
        token_count = max(1, len(tokens))

        # heurística: se maioria dos tokens são de 1 caractere, é o caso "a, m, o, r,  , ..."
        if single_char_count / token_count >= 0.6 and token_count >= 3:
            # reconstrói palavras a partir dos tokens-char
            words = reconstruct_from_char_tokens(tokens)
            return ", ".join(words) if words else s.strip()

        # caso normal: tokens representam palavras/phrases
        # limpa tokens vazios e junta
        cleaned = [t for t in tokens if t is not None and t.strip() != ""]
        if len(cleaned) > 1:
            return ", ".join(cleaned)
        # se não houve split por vírgula (apenas uma string), retorna trimmed
        return s.strip()

    # se for outro tipo (int/float), converte
    return str(value)

# Função que aplica ao JSON da linha (col_json) e retorna series com 3 campos
def parse_row_json(raw):
    try:
        if pd.isna(raw):
            return pd.Series([None, None, None])
    except:
        pass

    # limpa markdown e tenta carregar como JSON completo (objeto com keys)
    txt = clean_markdown_block(raw) if isinstance(raw, str) else raw

    # Se txt parecer um JSON object -> parse
    try:
        obj = json.loads(txt)
        # se obj é dicionário com as chaves esperadas
        if isinstance(obj, dict):
            temas = normalize_field(obj.get("temas_principales"))
            emocion_alegria = obj.get("emocion_alegria")
            emocion_tristeza = obj.get("emocion_tristeza")
            emocion_miedo = obj.get("emocion_miedo")
            emocion_ira = obj.get("emocion_ira")
            emocion_sorpresa = obj.get("emocion_sorpresa")
            emocion_asco = obj.get("emocion_asco")
            emocion_principal = obj.get("emocion_principal")
            resumen = obj.get("resumen")
            tono_general = obj.get("tono_general")
            energia_emocional = obj.get("energia_emocional")
            valencia = obj.get("valencia")
            activacion = obj.get("activacion")
            if isinstance(resumen, str):
                resumen = resumen.strip()
            return pd.Series([temas, emocion_alegria, emocion_tristeza, emocion_miedo, emocion_ira, emocion_sorpresa, emocion_asco, emocion_principal, resumen, tono_general, energia_emocional, valencia, activacion])
    except Exception:
        # não conseguiu parsear como JSON completo; seguir para tentativa manual
        pass

    # Se chegou aqui, tente tratar txt como string que contém os campos sequencialmente
    # Tentativa simples: procura por '"temas_principales":' etc usando regex
    try:
        # extrai conteúdo entre "temas_principales": (até próxima chave)
        m_t = re.search(r'"temas_principales"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_eal = re.search(r'"emocion_alegria"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_et = re.search(r'"emocion_tristeza"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_em = re.search(r'"emocion_miedo"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_ei = re.search(r'"emocion_ira"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_es= re.search(r'"emocion_sorpresa"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_eas = re.search(r'"emocion_asco"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_ep = re.search(r'"emocion_principal"\s*:\s*(\[[^\]]*\]|"[^"]*"|[^,}]+)', txt)
        m_r = re.search(r'"resumen"\s*:\s*(?:"([^"]*)"|\'([^\']*)\'|([^,}]+))', txt)
        m_to = re.search(r'"tono_general"\s*:\s*(?:"([^"]*)"|\'([^\']*)\'|([^,}]+))', txt)
        m_en = re.search(r'"energia_emocional"\s*:\s*(?:"([^"]*)"|\'([^\']*)\'|([^,}]+))', txt)
        m_v = re.search(r'"valencia"\s*:\s*(?:"([^"]*)"|\'([^\']*)\'|([^,}]+))', txt)
        m_a = re.search(r'"activacion"\s*:\s*(?:"([^"]*)"|\'([^\']*)\'|([^,}]+))', txt)

        temas_raw = m_t.group(1) if m_t else None
        emocion_alegria_raw = m_eal.group(1) if m_eal else None
        emocion_tristeza_raw = m_et.group(1) if m_et else None
        emocion_miedo_raw = m_em.group(1) if m_em else None
        emocion_ira_raw = m_ei.group(1) if m_ei else None
        emocion_sorpresa_raw = m_es.group(1) if m_es else None
        emocion_asco_raw = m_eas.group(1) if m_eas else None
        emocion_principal_raw = m_ep.group(1) if m_ep else None
        resumen_raw = None
        tono_general_raw = None
        energia_emocional_raw = None
        valencia_raw = m_v.group(1) if m_v else None
        activacion_raw = m_a.group(1) if m_a else None
        if m_r:
            resumen_raw = m_r.group(1) or m_r.group(2) or m_r.group(3)
        if m_to:
            tono_general_raw = m_to.group(1) or m_to.group(2) or m_to.group(3)
        if m_en:
            energia_emocional_raw = m_en.group(1) or m_en.group(2) or m_en.group(3)

        temas = normalize_field(temas_raw) if temas_raw is not None else None
        emocion_alegria = emocion_alegria_raw.strip() if emocion_alegria_raw is not None else None
        emocion_tristeza = emocion_tristeza_raw.strip() if emocion_tristeza_raw is not None else None
        emocion_miedo = emocion_miedo_raw.strip() if emocion_miedo_raw is not None else None
        emocion_ira = emocion_ira_raw.strip() if emocion_ira_raw is not None else None
        emocion_sorpresa = emocion_sorpresa_raw.strip() if emocion_sorpresa_raw is not None else None
        emocion_asco = emocion_asco_raw.strip() if emocion_asco_raw is not None else None
        emocion_principal = emocion_principal_raw.strip() if emocion_principal_raw is not None else None
        resumen = resumen_raw.strip() if isinstance(resumen_raw, str) else None
        tono_general = tono_general_raw.strip() if isinstance(tono_general_raw, str) else None
        energia_emocional = energia_emocional_raw.strip() if isinstance(energia_emocional_raw, str) else None
        valencia = valencia_raw.strip() if valencia_raw is not None else None
        activacion = activacion_raw.strip() if activacion_raw is not None else None

        return pd.Series([temas, emocion_alegria, emocion_tristeza, emocion_miedo, emocion_ira, emocion_sorpresa, emocion_asco, emocion_principal, resumen, tono_general, energia_emocional, valencia, activacion])
    except Exception:
        return pd.Series([None, None, None, None, None, None, None, None, None, None, None, None, None])



In [ ]:
df_resultado[["temas_principales", "emocion_alegria", "emocion_tristeza", "emocion_miedo", "emocion_ira", "emocion_sorpresa", "emocion_asco", "emocion_principal", "resumen", "tono_general", "energia_emocional", "valencia", "activacion"]] = df_resultado['erro_parse'].apply(parse_row_json)

In [ ]:
df_lyrics_analysis = df_lyrics_final.merge(df_resultado[['id', "temas_principales", "emocion_alegria", "emocion_tristeza", "emocion_miedo", "emocion_ira", "emocion_sorpresa", "emocion_asco", "emocion_principal", "resumen", "tono_general", "energia_emocional", "valencia", "activacion"]], left_on='id', right_on='id')

## Propriedades de AcousticBrainz

### Busca ID de la música en MusicBrainz

In [ ]:
musicbrainzngs.set_useragent("LyricsAnalysisApp", "1.0", "xxx")

In [ ]:
# Query MusicBrainz for MBID by ISRC and add 'mbid' column to df_lyrics_analysis.
# Respects 2s delay between requests to avoid API limits.

def _mbid_from_isrc(isrc):
    if pd.isna(isrc) or not str(isrc).strip():
        return None
    try:
        res = musicbrainzngs.get_recordings_by_isrc(str(isrc).strip())
        recs = res.get('isrc', {}).get('recording-list') or []
        if recs:
            return recs[0].get('id')
    except Exception as e:
        # opcional: log simples
        print(f"Error for ISRC {isrc}: {e}")
    return None

# Garantir coluna mbid existente
df_lyrics_analysis['mbid'] = None

for idx, row in df_lyrics_analysis.iterrows():
    isrc = row.get('isrc')
    mbid = _mbid_from_isrc(isrc)
    df_lyrics_analysis.at[idx, 'mbid'] = mbid
    print(f"[{idx}] ISRC={isrc} -> MBID={mbid}")
    time.sleep(2)  # delay para respeitar limites API

### Busca en AcousticBrainz

In [ ]:
mbid = "ac71eb2c-0a25-4f06-acff-13bec69d06e1"

# Endpoint "highlevel" (pode trocar por "lowlevel")
url = f"https://acousticbrainz.org/api/v1/{mbid}/high-level"

# Fazendo a requisição
response = requests.get(url)

# Convertendo para JSON
data = response.json()

# Exemplo: mostrar algumas informações
print("Gênero:", data["highlevel"]["genre_dortmund"]["value"])
print("Mood happy:", data["highlevel"]["mood_happy"]["value"])
print("Mood acoustic:", data["highlevel"]["mood_acoustic"]["value"])
print("Mood aggressive:", data["highlevel"]["mood_aggressive"]["value"])
print("Mood party:", data["highlevel"]["mood_party"]["value"])
print("Mood relaxed:", data["highlevel"]["mood_relaxed"]["value"])
print("Mood sad:", data["highlevel"]["mood_sad"]["value"])
print("Mood mirex:", data["highlevel"]["moods_mirex"]["value"])
print("Danceability", data["highlevel"]["danceability"]["value"])

In [ ]:
for idx, mbid in df_lyrics_analysis['mbid'].items():
    if pd.isna(mbid) or not str(mbid).strip():
        print(f"[{idx}] skipping empty MBID")
        continue

    url_high = f"https://acousticbrainz.org/api/v1/{mbid}/high-level"
    url_low = f"https://acousticbrainz.org/api/v1/{mbid}/low-level"

    try:
        response_high = requests.get(url_high, timeout=10)
        response_high.raise_for_status()
        data_high = response_high.json()
    except Exception as e:
        print(f"[{idx}] error fetching high-level for MBID={mbid}: {e}")
        data_high = {}

    try:
        response_low = requests.get(url_low, timeout=10)
        response_low.raise_for_status()
        data_low = response_low.json()
    except Exception as e:
        print(f"[{idx}] error fetching low-level for MBID={mbid}: {e}")
        data_low = {}

    # extract safely using .get chains
    genre_dortmund = data_high.get("highlevel", {}).get("genre_dortmund", {}).get("value")
    mood_happy = data_high.get("highlevel", {}).get("mood_happy", {}).get("value")
    mood_acoustic = data_high.get("highlevel", {}).get("mood_acoustic", {}).get("value")
    mood_aggressive = data_high.get("highlevel", {}).get("mood_aggressive", {}).get("value")
    mood_party = data_high.get("highlevel", {}).get("mood_party", {}).get("value")
    mood_relaxed = data_high.get("highlevel", {}).get("mood_relaxed", {}).get("value")
    mood_sad = data_high.get("highlevel", {}).get("mood_sad", {}).get("value")
    moods_mirex = data_high.get("highlevel", {}).get("moods_mirex", {}).get("value")
    danceability_high = data_high.get("highlevel", {}).get("danceability", {}).get("value")

    bpm = data_low.get("rhythm", {}).get("bpm")
    danceability_low = data_low.get("rhythm", {}).get("danceability")

    # assign values only to the current row
    df_lyrics_analysis.at[idx, 'genre_dortmund'] = genre_dortmund
    df_lyrics_analysis.at[idx, 'mood_happy'] = mood_happy
    df_lyrics_analysis.at[idx, 'mood_acoustic'] = mood_acoustic
    df_lyrics_analysis.at[idx, 'mood_aggressive'] = mood_aggressive
    df_lyrics_analysis.at[idx, 'mood_party'] = mood_party
    df_lyrics_analysis.at[idx, 'mood_relaxed'] = mood_relaxed
    df_lyrics_analysis.at[idx, 'mood_sad'] = mood_sad
    df_lyrics_analysis.at[idx, 'moods_mirex'] = moods_mirex
    df_lyrics_analysis.at[idx, 'danceability_high'] = danceability_high
    df_lyrics_analysis.at[idx, 'bpm'] = bpm
    df_lyrics_analysis.at[idx, 'danceability_low'] = danceability_low

    print(f"[{idx}] MBID={mbid} -> genre={genre_dortmund}, bpm={bpm}")
    time.sleep(2)  # delay para respeitar limites API

In [ ]:
df_lyrics_analysis

### Utiliza Essentia para buscar bpm y danceability de canciones no encontradas en AcousticBrainz

In [ ]:
yaml_folder = r"musicas\yaml"

for y in os.listdir(yaml_folder):
    yaml_file = os.path.join(yaml_folder, y)
    print(y)
    try:
        with open(yaml_file, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)

        bpm = data["rhythm"]["bpm"]
        danceability = data["rhythm"]["danceability"]
        albumartist = data["metadata"]["tags"]["albumartist"][0]
        title = data["metadata"]["tags"]["title"][0]
        track = data["metadata"]["tags"]["tracknumber"][0]
        tracknumber = int(track.split("/")[0]) if "/" in track else int(track)

        mask = (
            (df_lyrics_analysis['diva_name'].str.lower().str.strip() == albumartist.lower().strip()) &
            (df_lyrics_analysis['name_modified'].str.lower().str.strip() == title.lower().strip()) &
            (df_lyrics_analysis['track_number'] == tracknumber)
        )

        if mask.any():
            df_lyrics_analysis.loc[mask, 'bpm'] = bpm
            df_lyrics_analysis.loc[mask, 'danceability_low'] = danceability
            print('done')
        else:
            print(f'no match for {y}: albumartist="{albumartist}", title="{title}", track={tracknumber}')
    except Exception as e:
        print(f"Error processing file {y}: {e}")

In [ ]:
def extract_essentia_features(essentia_exe, input_audio, output_yaml):
    result = subprocess.run(
        [essentia_exe, input_audio, output_yaml],
        cwd=str(Path(essentia_exe).parent),
        capture_output=True,
        text=True
    )
    return result

In [ ]:
ESSENTIA_EXE = r"essentia_streaming_extractor_music.exe"
musicas_folder = r"\musicas"
processados_folder = r"\musicas\processados"
yaml_folder = r"\musicas\yaml"

# List all audio files
audio_extensions = ('.mp3', '.wav', '.flac', '.aac', '.ogg', '.m4a')
audio_files = [file for file in os.listdir(musicas_folder) if file.lower().endswith(audio_extensions)]

for audio_file in audio_files:
    print(audio_file)

    normalize_filename = audio_file.replace("’", "'").replace("í", "i").replace("á", "a").replace("é", "e").replace("ó", "o").replace("ú", "u").replace("Á", "A").replace("É", "E").replace("Í", "I").replace("Ó", "O").replace("Ú", "U").replace("ñ", "n").replace("Ñ", "N").replace("ä", "a").replace("Ä", "A").replace("ö", "o").replace("Ö", "O").replace("ü", "u").replace("Ü", "U").replace("ß", "ss").replace("★", "_").replace("…", "...")
    audio_path = os.path.join(musicas_folder, audio_file)
    output_yaml = 'C:\\essentia\\output\\' + normalize_filename + '.yaml'

    result = extract_essentia_features(ESSENTIA_EXE, audio_path, output_yaml)

    with open(output_yaml, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

        bpm = data["rhythm"]["bpm"]
        danceability = data["rhythm"]["danceability"]
        albumartist = data["metadata"]["tags"]["albumartist"][0]
        title = data["metadata"]["tags"]["title"][0]
        track = data["metadata"]["tags"]["tracknumber"][0]
        tracknumber = int(track.split("/")[0]) if "/" in track else int(track)

    mask = (df_lyrics_analysis['diva_name'].str.lower().str.strip() == albumartist.lower().strip()) & (df_lyrics_analysis['name_modified'].str.lower().str.strip() == title.lower().strip()) & (df_lyrics_analysis['track_number'] == tracknumber)

    df_lyrics_analysis.loc[mask, 'bpm'] = bpm
    df_lyrics_analysis.loc[mask, 'danceability_low'] = danceability

    
    #move processed file to processed folder
    shutil.move(audio_path, os.path.join(processados_folder, audio_file))

    #move yaml file to yaml folder
    shutil.move(output_yaml, os.path.join(yaml_folder, audio_file + '.yaml'))


    print('Return code:', result.returncode, ' BPM:', bpm, ' Danceability:', danceability)
    print('')